# Module 1 Homework — Stock Markets Analytics Zoomcamp 2026

Lesson reference: [`01-intro-and-data-sources/Module01_Colab_Introduction_and_Data_Sources.ipynb`](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/01-intro-and-data-sources/Module01_Colab_Introduction_and_Data_Sources.ipynb)

Homework questions: https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw01


## Question 1: S&P 500 Stocks Added to the Index

**Main task:** Which year had the highest number of additions (starting from 2020)?

**Additional task:** How many current S&P 500 stocks have been in the index for more than 20 years?

Source: Wikipedia's "List of S&P 500 companies" page, scraped with `pandas.read_html()`.


In [1]:
"""
Question 1: S&P 500 Stocks Added to the Index

Main Task: Which year had the highest number of additions (starting from 2020)?
Additional Task: How many current S&P 500 stocks have been in the index for
more than 20 years?

Note: Wikipedia's "List of S&P 500 companies" page used to also carry a
separate "Selected changes to the list of components" history table, but as
of this run only the current-constituents table (with a "Date added" column)
is present, so that's the one we use.
"""
import io

import pandas as pd
import requests

WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"


def load_sp500_current():
    # can't call pd.read_html() directly on this URL (403) -- same trick as
    # the course notebook's "Web Scraping for Macro" section: fetch with a
    # browser-like User-Agent first, then hand the HTML to pandas.
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
    }
    resp = requests.get(WIKI_URL, headers=headers, timeout=30)
    resp.raise_for_status()
    tables = pd.read_html(io.StringIO(resp.text))
    current = tables[0]  # current constituents, has "Date added"
    return current


def main():
    current = load_sp500_current().copy()
    current["Date added"] = pd.to_datetime(current["Date added"], errors="coerce")
    current["year_added"] = current["Date added"].dt.year

    # --- Main task: additions per year from 2020 onward ---
    additions_by_year = (
        current[current["year_added"] >= 2020]
        .groupby("year_added")
        .size()
        .sort_index()
    )
    print("Additions by year (2020+):")
    print(additions_by_year)

    top_year = additions_by_year.idxmax()
    top_count = additions_by_year.max()
    print(f"\nANSWER Q1 (main): Year with most additions = {top_year} ({top_count} additions)")

    # --- Additional task: how many current stocks in index > 20 years ---
    today = pd.Timestamp.today()
    current["years_in_index"] = (today - current["Date added"]).dt.days / 365.25
    over_20 = current[current["years_in_index"] > 20]
    print(f"\nANSWER Q1 (additional): Stocks in index for more than 20 years = {len(over_20)}")
    print(
        over_20[["Symbol", "Security", "Date added", "years_in_index"]]
        .sort_values("years_in_index", ascending=False)
        .to_string(index=False)
    )


main()


Additions by year (2020+):
year_added
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
2026    13
dtype: int64

ANSWER Q1 (main): Year with most additions = 2025 (18 additions)

ANSWER Q1 (additional): Stocks in index for more than 20 years = 224
Symbol                           Security Date added  years_in_index
   MMM                                 3M 1957-03-04       69.530459
     F                 Ford Motor Company 1957-03-04       69.530459
    BA                             Boeing 1957-03-04       69.530459
   IBM                                IBM 1957-03-04       69.530459
   BMY               Bristol Myers Squibb 1957-03-04       69.530459
   HON             Honeywell Technologies 1957-03-04       69.530459
   HSY              Hershey Company (The) 1957-03-04       69.530459
   HIG                     Hartford (The) 1957-03-04       69.530459
   HAL                        Halliburton 1957-03-04       69.530459
   CAT                   Caterpillar Inc. 1957

## Question 2: Macro Indexes YTD (as of August 21, 2026)

**Main task:** How many indexes (out of 10) have better YTD returns than the US (S&P 500) as of August 21, 2026?

**Additional task:** Compare 3, 5, and 10-year performance trends.

Source: Yahoo Finance via `yfinance`.


In [2]:
"""
Question 2: Macro Indexes YTD (as of August 21, 2026)

Main Task: How many indexes (out of 10) have better YTD returns than the US
(S&P 500) as of August 21, 2026?
Additional Task: Compare 3, 5, and 10-year performance trends.
"""
import yfinance as yf
import pandas as pd

TICKERS = {
    "S&P 500 (US)": "^GSPC",
    "Shanghai Composite (China)": "000001.SS",
    "Hang Seng (Hong Kong)": "^HSI",
    "S&P/ASX 200 (Australia)": "^AXJO",
    "Nifty 50 (India)": "^NSEI",
    "S&P/TSX Composite (Canada)": "^GSPTSE",
    "DAX (Germany)": "^GDAXI",
    "FTSE 100 (UK)": "^FTSE",
    "Nikkei 225 (Japan)": "^N225",
    "IPC Mexico": "^MXX",
    "Ibovespa (Brazil)": "^BVSP",
}

START_DATE = "2026-01-01"
END_DATE = "2026-08-21"


def ytd_return(ticker, start_date=START_DATE, end_date=END_DATE):
    # yfinance end_date is exclusive, so add a day to include Aug 21 close.
    end_inclusive = (pd.Timestamp(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    df = yf.download(ticker, start=start_date, end=end_inclusive, progress=False, auto_adjust=False)
    if df.empty:
        return None
    close = df["Close"].dropna()
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    first_close = close.iloc[0]
    last_close = close.iloc[-1]
    ret = (last_close / first_close - 1) * 100
    return float(ret), close.index[0], close.index[-1]


def multi_year_return(ticker, years, end_date=END_DATE):
    end_ts = pd.Timestamp(end_date)
    start_ts = end_ts - pd.DateOffset(years=years)
    end_inclusive = (end_ts + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    df = yf.download(ticker, start=start_ts.strftime("%Y-%m-%d"), end=end_inclusive, progress=False, auto_adjust=False)
    if df.empty:
        return None
    close = df["Close"].dropna()
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    return float((close.iloc[-1] / close.iloc[0] - 1) * 100)


def main():
    print(f"YTD ({START_DATE} to {END_DATE}) returns:\n")
    results = {}
    for name, ticker in TICKERS.items():
        r = ytd_return(ticker)
        if r is None:
            print(f"{name:35s} ({ticker:10s}): NO DATA")
            continue
        ret, first_dt, last_dt = r
        results[name] = ret
        print(f"{name:35s} ({ticker:10s}): {ret:7.2f}%   [{first_dt.date()} -> {last_dt.date()}]")

    us_ret = results["S&P 500 (US)"]
    others = {k: v for k, v in results.items() if k != "S&P 500 (US)"}
    better = {k: v for k, v in others.items() if v > us_ret}

    print(f"\nS&P 500 YTD return: {us_ret:.2f}%")
    print(f"Indexes beating the S&P 500 (out of {len(others)}): {len(better)}")
    for k, v in sorted(better.items(), key=lambda kv: -kv[1]):
        print(f"  {k}: {v:.2f}%")

    print(f"\nANSWER Q2 (main): {len(better)} indexes out of {len(others)} beat the S&P 500 YTD.")

    # --- Additional task: 3, 5, 10-year trends ---
    print("\n--- Multi-year performance trends ---")
    for years in (3, 5, 10):
        print(f"\n{years}-year returns (through {END_DATE}):")
        multi = {}
        for name, ticker in TICKERS.items():
            r = multi_year_return(ticker, years)
            multi[name] = r
            if r is not None:
                print(f"  {name:35s}: {r:8.2f}%")
            else:
                print(f"  {name:35s}: NO DATA")
        us_multi = multi.get("S&P 500 (US)")
        if us_multi is not None:
            beat = [k for k, v in multi.items() if v is not None and k != "S&P 500 (US)" and v > us_multi]
            print(f"  -> Indexes beating S&P 500 over {years}y: {len(beat)} ({', '.join(beat) if beat else 'none'})")


main()


YTD (2026-01-01 to 2026-08-21) returns:



S&P 500 (US)                        (^GSPC     ):   11.90%   [2026-01-02 -> 2026-08-21]


Shanghai Composite (China)          (000001.SS ):   -2.94%   [2026-01-05 -> 2026-08-21]


Hang Seng (Hong Kong)               (^HSI      ):   -1.25%   [2026-01-02 -> 2026-08-21]


S&P/ASX 200 (Australia)             (^AXJO     ):    3.79%   [2026-01-02 -> 2026-08-21]


Nifty 50 (India)                    (^NSEI     ):   -7.25%   [2026-01-01 -> 2026-08-21]


... (35 outputs left out)

  FTSE 100 (UK)                      :    58.40%


  Nikkei 225 (Japan)                 :   297.73%


  IPC Mexico                         :    36.10%


  Ibovespa (Brazil)                  :   196.00%
  -> Indexes beating S&P 500 over 10y: 1 (Nikkei 225 (Japan))


## Question 3: S&P 500 Market Corrections Analysis

**Main task:** Calculate the median drawdown (in %) of significant market corrections (>= 5% decline from a running all-time high) in the S&P 500 index, using daily data from 1950 to present.


In [3]:
"""
Question 3: S&P 500 Market Corrections Analysis

Main Task: Calculate the median drawdown (in %) of significant market
corrections in the S&P 500 index.

Definition: a correction = a decline of at least 5% from a running
all-time high (peak) to the subsequent trough before a new all-time high
is made.
"""
import numpy as np
import pandas as pd
import yfinance as yf

TICKER = "^GSPC"
START_DATE = "1950-01-01"
MIN_DRAWDOWN_PCT = 5.0  # correction threshold


def get_sp500_history(start=START_DATE):
    df = yf.download(TICKER, start=start, progress=False, auto_adjust=False)
    close = df["Close"].dropna()
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    return close


def find_drawdown_episodes(close: pd.Series):
    """Identify peak-to-trough episodes relative to the running all-time high.

    A new episode starts every time the series makes a new all-time high and
    then declines; it ends when a new all-time high is reached (or at the
    end of the series, for an ongoing/unresolved drawdown).
    """
    episodes = []

    running_peak = close.iloc[0]
    episode_peak = close.iloc[0]
    episode_peak_date = close.index[0]
    episode_trough = close.iloc[0]
    episode_trough_date = close.index[0]
    in_episode = False

    for date, price in close.items():
        if price >= running_peak:
            if in_episode:
                dd_pct = (episode_trough / episode_peak - 1) * 100
                duration_days = (episode_trough_date - episode_peak_date).days
                episodes.append(
                    {
                        "peak_date": episode_peak_date,
                        "peak_price": episode_peak,
                        "trough_date": episode_trough_date,
                        "trough_price": episode_trough,
                        "drawdown_pct": dd_pct,
                        "duration_days": duration_days,
                        "resolved": True,
                    }
                )
                in_episode = False
            running_peak = price
            episode_peak = price
            episode_peak_date = date
            episode_trough = price
            episode_trough_date = date
        else:
            in_episode = True
            if price < episode_trough:
                episode_trough = price
                episode_trough_date = date

    if in_episode:
        dd_pct = (episode_trough / episode_peak - 1) * 100
        duration_days = (episode_trough_date - episode_peak_date).days
        episodes.append(
            {
                "peak_date": episode_peak_date,
                "peak_price": episode_peak,
                "trough_date": episode_trough_date,
                "trough_price": episode_trough,
                "drawdown_pct": dd_pct,
                "duration_days": duration_days,
                "resolved": False,
            }
        )

    return pd.DataFrame(episodes)


def main():
    close = get_sp500_history()
    print(f"S&P 500 daily closes: {close.index[0].date()} -> {close.index[-1].date()} ({len(close)} rows)")

    episodes = find_drawdown_episodes(close)
    corrections = episodes[episodes["drawdown_pct"] <= -MIN_DRAWDOWN_PCT].copy()
    corrections["abs_drawdown_pct"] = corrections["drawdown_pct"].abs()

    print(f"\nTotal peak-to-trough episodes found: {len(episodes)}")
    print(f"Episodes with drawdown >= {MIN_DRAWDOWN_PCT}%: {len(corrections)}")

    print("\nAll qualifying corrections (peak date, trough date, drawdown %, duration days):")
    print(
        corrections[["peak_date", "trough_date", "drawdown_pct", "duration_days", "resolved"]]
        .sort_values("peak_date")
        .to_string(index=False)
    )

    p25, p50, p75 = np.percentile(corrections["abs_drawdown_pct"], [25, 50, 75])
    print(f"\n25th percentile drawdown: {p25:.2f}%")
    print(f"50th percentile (median) drawdown: {p50:.2f}%")
    print(f"75th percentile drawdown: {p75:.2f}%")

    dur25, dur50, dur75 = np.percentile(corrections["duration_days"], [25, 50, 75])
    print(f"\n25th percentile duration: {dur25:.1f} days")
    print(f"50th percentile (median) duration: {dur50:.1f} days")
    print(f"75th percentile duration: {dur75:.1f} days")

    print(f"\nANSWER Q3: Median drawdown of significant corrections = {p50:.2f}%")


main()


S&P 500 daily closes: 1950-01-03 -> 2026-09-14 (19296 rows)

Total peak-to-trough episodes found: 682
Episodes with drawdown >= 5.0%: 74

All qualifying corrections (peak date, trough date, drawdown %, duration days):
 peak_date trough_date  drawdown_pct  duration_days  resolved
1950-06-12  1950-07-17    -14.020615             35      True
1950-11-24  1950-12-04     -6.496062             10      True
1951-05-03  1951-06-29     -8.110480             57      True
1951-10-15  1951-11-23     -6.079668             39      True
1952-01-22  1952-02-20     -6.366584             29      True
1952-08-08  1952-10-22     -6.849315             75      True
1953-01-05  1953-09-14    -14.816207            252      True
1955-01-03  1955-01-17     -5.904757             14      True
1955-03-04  1955-03-14     -6.823031             10      True
1955-09-23  1955-10-11    -10.585145             18      True
1955-11-14  1956-01-23     -7.110535             70      True
1956-03-20  1956-05-28     -9.760590  

## Question 4: Earnings Surprise Analysis for Amazon (AMZN)

**Main task:** Calculate the median 2-day percentage change in stock prices following positive earnings surprise days.

**Additional task:** Examine correlation between surprise magnitude and price reaction; assess bull vs. bear market differences.

Source: `yfinance`'s `Ticker.get_earnings_dates()` plus daily price history.


In [4]:
"""
Question 4: Earnings Surprise Analysis for Amazon (AMZN)

Main Task: Calculate the median 2-day percentage change in stock prices
following positive earnings surprise days.
Additional Task: Examine correlation between surprise magnitude and price
reaction; assess bull vs bear market differences.

Methodology: for each earnings date, take the 3-day trading-day sequence
[day before the announcement, the announcement day, the day after] and
define the "2-day return" as the % change from the close the day before
the announcement to the close the day after (a 2-trading-day span that
straddles the earnings event).
"""
import numpy as np
import pandas as pd
import yfinance as yf

TICKER = "AMZN"
SINCE = "2020-10-29"  # homework spec: "25 entries from 2020-10-29"


def get_earnings_surprises(ticker=TICKER, limit=30, since=SINCE):
    tk = yf.Ticker(ticker)
    df = tk.get_earnings_dates(limit=limit)
    df = df.dropna(subset=["Surprise(%)"]).sort_index()
    if since is not None:
        df = df[df.index >= pd.Timestamp(since, tz=df.index.tz)]
    return df


def get_price_history(ticker=TICKER, start="2020-01-01"):
    df = yf.download(ticker, start=start, progress=False, auto_adjust=False)
    close = df["Close"].dropna()
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    return close


def two_day_return_around(close: pd.Series, earnings_ts: pd.Timestamp):
    """3-day sequence: [day-before, announcement day, day-after].
    2-day return = close(day-after) / close(day-before) - 1.
    """
    earnings_date = pd.Timestamp(earnings_ts).tz_localize(None).normalize()
    idx = close.index
    pos = idx.searchsorted(earnings_date)  # index of the announcement day (or next trading day)
    if pos - 1 < 0 or pos + 1 >= len(idx):
        return None
    day_before, announce_day, day_after = idx[pos - 1], idx[pos], idx[pos + 1]
    ret_pct = (close.iloc[pos + 1] / close.iloc[pos - 1] - 1) * 100
    return ret_pct, day_before, announce_day, day_after


def main():
    earnings = get_earnings_surprises()
    print(f"Earnings entries with surprise data (since {SINCE}): {len(earnings)}")
    print(earnings[["Reported EPS", "EPS Estimate", "Surprise(%)"]].to_string())

    close = get_price_history(start="2020-01-01")
    print(f"\nPrice history: {close.index[0].date()} -> {close.index[-1].date()} ({len(close)} rows)")

    records = []
    for ts, row in earnings.iterrows():
        result = two_day_return_around(close, ts)
        if result is None:
            continue
        ret_pct, d_before, d_announce, d_after = result
        records.append(
            {
                "earnings_date": pd.Timestamp(ts).tz_localize(None),
                "surprise_pct": row["Surprise(%)"],
                "day_before": d_before,
                "announce_day": d_announce,
                "day_after": d_after,
                "two_day_return_pct": ret_pct,
            }
        )

    res = pd.DataFrame(records).sort_values("earnings_date")
    print("\nPer-earnings 2-day returns (day-before -> day-after close):")
    print(res.to_string(index=False))

    positive = res[res["surprise_pct"] > 0]
    print(f"\nPositive-surprise earnings events: {len(positive)} / {len(res)}")

    median_2day = positive["two_day_return_pct"].median()
    print(f"\nANSWER Q4 (main): Median 2-day % change after positive surprises = {median_2day:.2f}%")

    # --- Additional task: correlation between surprise magnitude and reaction ---
    corr_all = res["surprise_pct"].corr(res["two_day_return_pct"])
    corr_pos = positive["surprise_pct"].corr(positive["two_day_return_pct"])
    print(f"\nCorrelation (surprise % vs 2-day return), all events: {corr_all:.3f}")
    print(f"Correlation (surprise % vs 2-day return), positive-surprise events only: {corr_pos:.3f}")

    # crude bull/bear split: 2022 was a broad bear market for growth/tech names
    bear_mask = (res["earnings_date"] >= "2022-01-01") & (res["earnings_date"] <= "2022-12-31")
    bear = res[bear_mask]
    bull = res[~bear_mask]
    print(f"\n'Bear market' (2022) events: {len(bear)}, median 2-day return = {bear['two_day_return_pct'].median():.2f}%")
    print(f"'Bull/other' events: {len(bull)}, median 2-day return = {bull['two_day_return_pct'].median():.2f}%")


main()


Earnings entries with surprise data (since 2020-10-29): 24
                           Reported EPS  EPS Estimate  Surprise(%)
Earnings Date                                                     
2020-10-29 16:00:00-04:00          0.62          0.37        66.17
2021-02-02 16:00:00-05:00          0.70          0.36        97.39
2021-04-29 16:00:00-04:00          0.79          0.48        64.11
2021-07-29 16:00:00-04:00          0.76          0.62        22.57
2021-10-28 16:00:00-04:00          0.31          0.44       -30.85
2022-02-03 16:00:00-05:00          1.39          0.19       641.94
2022-04-28 16:00:00-04:00         -0.38          0.41      -191.55
2022-07-28 16:00:00-04:00         -0.20          0.12      -273.30
2022-10-27 16:00:00-04:00          0.28          0.21        33.00
2023-02-02 16:00:00-05:00          0.03          0.16       -81.68
2023-04-27 16:00:00-04:00          0.31          0.21        46.82
2023-08-03 16:00:00-04:00          0.65          0.34        91.19
202


Price history: 2020-01-02 -> 2026-09-14 (1683 rows)

Per-earnings 2-day returns (day-before -> day-after close):
      earnings_date  surprise_pct day_before announce_day  day_after  two_day_return_pct
2020-10-29 16:00:00         66.17 2020-10-28   2020-10-29 2020-10-30           -4.003764
2021-02-02 16:00:00         97.39 2021-02-01   2021-02-02 2021-02-03           -0.907901
2021-04-29 16:00:00         64.11 2021-04-28   2021-04-29 2021-04-30            0.257915
2021-07-29 16:00:00         22.57 2021-07-28   2021-07-29 2021-07-30           -8.338937
2021-10-28 16:00:00        -30.85 2021-10-27   2021-10-28 2021-10-29           -0.591301
2022-02-03 16:00:00        641.94 2022-02-02   2022-02-03 2022-02-04            4.665611
2022-04-28 16:00:00       -191.55 2022-04-27   2022-04-28 2022-04-29          -10.049798
2022-07-28 16:00:00       -273.30 2022-07-27   2022-07-28 2022-07-29           11.556581
2022-10-27 16:00:00         33.00 2022-10-26   2022-10-27 2022-10-28          -10.591

## Question 5: Capstone Project Brainstorm (Optional)

Free-text: describe your capstone project idea (asset class, geography, industry focus, investment horizon, proposed methodology). Fill in `q5.py` with your own answer.


In [5]:
"""
Question 5: Capstone Project Brainstorm (Optional)

Free-text answer describing your capstone project idea, including asset
class, geography, industry focus, investment horizon, and proposed
methodology.
"""

ANSWER = """
TODO: fill in with your own capstone project idea -- asset class,
geography, industry focus, investment horizon, and proposed methodology.
"""


def main():
    print(ANSWER.strip())


main()


TODO: fill in with your own capstone project idea -- asset class,
geography, industry focus, investment horizon, and proposed methodology.


## Question 6: New Metrics Investigation (Optional)

Free-text: identify additional metrics/time series relevant to your project. Fill in `q6.py` with your own answer.


In [6]:
"""
Question 6: New Metrics Investigation (Optional)

Free-text answer identifying additional metrics or time series relevant to
your project, explaining their value and retrieval methods.
"""

ANSWER = """
TODO: fill in with additional metrics/time series relevant to your
project, why they matter, and how you'd retrieve them.
"""


def main():
    print(ANSWER.strip())


main()


TODO: fill in with additional metrics/time series relevant to your
project, why they matter, and how you'd retrieve them.


## Summary of Answers

| Question | Answer |
|---|---|
| Q1 — Year with most S&P 500 additions (2020+) | **2025** (18 additions) |
| Q1 (extra) — Current stocks in index > 20 years | 224 |
| Q2 — Indexes beating S&P 500 YTD (of 10) | **2** (Nikkei 225, S&P/TSX Composite) |
| Q3 — Median drawdown of significant corrections | **~8%** (7.99%) |
| Q4 — Median 2-day return after positive earnings surprise (AMZN) | **0.35%** |
| Q5 — Capstone project brainstorm | free text — see `q5.py` |
| Q6 — New metrics investigation | free text — see `q6.py` |
